# 🚀 Qwen3-0.6B LoRA Fine-tuning — Kaggle 2×T4 GPUs (5 Epochs)

**Optimized for Kaggle with 2× NVIDIA T4 (16 GB VRAM each):**
- FP16 precision (T4 does not support BF16)
- Batch size 8/GPU × 2 GPUs × 4 grad_accum = 64 effective
- Dynamic padding, gradient checkpointing, group_by_length
- Auto-handles DataParallel for multi-GPU
- Fits within Kaggle's 9-hour GPU time limit

**Instructions:**
1. Upload your 3 CSV files as a **Kaggle Dataset**
2. Add that dataset to this notebook
3. Change `KAGGLE_INPUT_DIR` in Cell 2 to match your dataset path
4. Select **GPU T4 x2** accelerator
5. Run all cells

In [ ]:
# Install / upgrade required libraries
!pip install -q pandas numpy torch tqdm accelerate sentencepiece rouge_score evaluate nltk
!pip install -q -U transformers datasets peft

## Cell 0: Environment Setup

In [1]:
print("=" * 60)
print("CELL 0: ENVIRONMENT SETUP")
print("=" * 60)

import torch
print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
n_gpus = torch.cuda.device_count()
print(f"Number of GPUs: {n_gpus}")
for i in range(n_gpus):
    props = torch.cuda.get_device_properties(i)
    print(f"  GPU {i}: {props.name} — {props.total_memory / 1024**3:.1f} GiB")

print("CELL 0 DONE\n")

CELL 0: ENVIRONMENT SETUP
PyTorch: 2.8.0+cu126
CUDA available: True
Number of GPUs: 2
  GPU 0: Tesla T4 — 14.6 GiB
  GPU 1: Tesla T4 — 14.6 GiB
CELL 0 DONE



## Cell 1: Imports

In [2]:
print("=" * 60)
print("CELL 1: IMPORTS")
print("=" * 60)

import os
import shutil
import time
import json
import gc
import traceback

import pandas as pd
import numpy as np
import evaluate
from datasets import Dataset, load_from_disk
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    TrainingArguments,
    Trainer,
    GenerationConfig,
)
from peft import LoraConfig, get_peft_model, TaskType, PeftModel
import nltk
from tqdm.auto import tqdm

try:
    nltk.data.find('tokenizers/punkt')
except LookupError:
    nltk.download('punkt', quiet=True)
try:
    nltk.data.find('tokenizers/punkt_tab')
except LookupError:
    nltk.download('punkt_tab', quiet=True)

print("CELL 1 DONE\n")

CELL 1: IMPORTS
CELL 1 DONE



## Cell 2: Configuration

In [ ]:
print("=" * 60)
print("CELL 2: CONFIGURATION")
print("=" * 60)

# =============================================================
# ★★★ CHANGE THIS to your Kaggle dataset path ★★★
# Upload your 3 CSV files as a Kaggle dataset, then set the path.
# Example: /kaggle/input/advertisement-training-data
# =============================================================
KAGGLE_INPUT_DIR = 'Your Input Dir'

TRAIN_CSV_PATH = os.path.join(KAGGLE_INPUT_DIR, 'Final_advertisement_filtered_adv_lte1024_train_90.csv')
VAL_CSV_PATH   = os.path.join(KAGGLE_INPUT_DIR, 'Final_advertisement_filtered_adv_lte1024_val_5.csv')
TEST_CSV_PATH  = os.path.join(KAGGLE_INPUT_DIR, 'Final_advertisement_filtered_adv_lte1024_test_5.csv')

# Local working directory (fast SSD on Kaggle)
LOCAL_WORK_DIR = '/kaggle/working'
LOCAL_DATA_DIR = os.path.join(LOCAL_WORK_DIR, 'data_cache')

# Output paths — all in /kaggle/working so they persist after commit
OUTPUT_DIR           = os.path.join(LOCAL_WORK_DIR, 'output')
CHECKPOINT_DIR       = os.path.join(OUTPUT_DIR, 'checkpoints')
FINAL_MODEL_DIR      = os.path.join(OUTPUT_DIR, 'final_best_model')
TOKENIZER_SAVE_DIR   = os.path.join(OUTPUT_DIR, 'tokenizer')

# ---- Model ----
MODEL_NAME = "Qwen/Qwen3-0.6B"

# ---- LoRA ----
LORA_R = 8
LORA_ALPHA = 16
LORA_DROPOUT = 0.05
LORA_TARGET_MODULES = [
    "q_proj", "k_proj", "v_proj", "o_proj",
    "gate_proj", "up_proj", "down_proj",
]

# ---- Training hyper-params (tuned for 2×T4 16 GB each) ----
NUM_TRAIN_EPOCHS = 0.5
PER_DEVICE_TRAIN_BATCH_SIZE = 2        # per GPU → 8×2 GPUs = 16 samples/step
GRADIENT_ACCUMULATION_STEPS = 8        # effective batch = 16 × 4 = 64
PER_DEVICE_EVAL_BATCH_SIZE = 4
LEARNING_RATE = 2e-4
WEIGHT_DECAY = 0.01
WARMUP_RATIO = 0.03
MAX_SEQ_LENGTH = 2048

# T4 does NOT support BF16 — use FP16
COMPUTE_DTYPE = torch.float16

# Detect optimizer availability
OPTIM = "adamw_torch"
if torch.__version__ >= "2.0":
    try:
        # Test if fused optimizer works on this device
        _test = torch.optim.AdamW([torch.zeros(1, device='cuda')], fused=True)
        OPTIM = "adamw_torch_fused"
        del _test
    except Exception:
        pass

# Create directories
for d in [LOCAL_DATA_DIR, OUTPUT_DIR, CHECKPOINT_DIR, FINAL_MODEL_DIR, TOKENIZER_SAVE_DIR]:
    os.makedirs(d, exist_ok=True)

print(f"  Model:             {MODEL_NAME}")
print(f"  GPUs:              {n_gpus}× T4")
print(f"  Epochs:            {NUM_TRAIN_EPOCHS}")
print(f"  Batch (per GPU):   {PER_DEVICE_TRAIN_BATCH_SIZE}")
print(f"  Grad accum:        {GRADIENT_ACCUMULATION_STEPS}")
print(f"  Effective batch:   {PER_DEVICE_TRAIN_BATCH_SIZE * n_gpus * GRADIENT_ACCUMULATION_STEPS}")
print(f"  LR:                {LEARNING_RATE}")
print(f"  Max seq length:    {MAX_SEQ_LENGTH}")
print(f"  Compute dtype:     {COMPUTE_DTYPE}")
print(f"  Optimizer:         {OPTIM}")
print(f"  LoRA r={LORA_R}, alpha={LORA_ALPHA}, dropout={LORA_DROPOUT}")
print(f"  LoRA targets:      {LORA_TARGET_MODULES}")
print(f"  Input data:        {KAGGLE_INPUT_DIR}")
print(f"  Output:            {OUTPUT_DIR}")
print("CELL 2 DONE\n")

CELL 2: CONFIGURATION
  Model:             Qwen/Qwen3-0.6B
  GPUs:              2× T4
  Epochs:            0.5
  Batch (per GPU):   2
  Grad accum:        8
  Effective batch:   32
  LR:                0.0002
  Max seq length:    2048
  Compute dtype:     torch.float16
  Optimizer:         adamw_torch_fused
  LoRA r=8, alpha=16, dropout=0.05
  LoRA targets:      ['q_proj', 'k_proj', 'v_proj', 'o_proj', 'gate_proj', 'up_proj', 'down_proj']
  Input data:        /kaggle/input/datasets/minhlinn/advertisement
  Output:            /kaggle/working/output
CELL 2 DONE



## Cell 3: Tokenizer Preparation

In [4]:
print("=" * 60)
print("CELL 3: TOKENIZER PREPARATION")
print("=" * 60)

# --- Custom tokens (emojis/symbols used in advertisements) ---
VALID_ENDINGS = ('.', '!', '?', '…', '#️⃣', '1️⃣', '2️⃣', '3️⃣', '4️⃣', '®', '™', '⌚', '⌨️', '⏩', '⏰', '⏱️', '⏳', '⏸️', '☀️', '☁️', '☂️', '☃️', '☄️', '☎', '☎️', '☑️', '☔', '☕', '☘️', '☝️', '☹️', '☺️', '♟️', '♠️', '♻️', '⚒️', '⚓', '⚔️', '⚕️', '⚖️', '⚙️', '⚠️', '⚡', '⚫', '⚽', '⚾', '⛈️', '⛔', '⛰️', '⛱️', '⛳', '⛵', '⛷️', '⛹️♂️', '⛺', '⛽', '✂️', '✅', '✈️', '✉️', '✊', '✋', '✌️', '✍️', '✏️', '✒️', '✔', '✔️', '✖', '✝️', '✨', '❄️', '❌', '❓', '❗', '❣️', '❤', '❤️', '➕', '➖', '➡️', '⬅️', '⬆️', '⭐', '🀄', '🃏', '🅿️', '🆒', '🆕', '🆘', '🇦🇷', '🇦🇺', '🇧🇷', '🇨🇦', '🇨🇳', '🇩🇪', '🇩🇰', '🇪🇸', '🇫🇮', '🇫🇷', '🇬🇧', '🇮🇪', '🇮🇹', '🇯🇵', '🇰🇷', '🇲🇽', '🇳🇿', '🇸🇪', '🇸🇬', '🇹🇭', '🇹🇼', '🇺🇸', '🇻🇳', '🌀', '🌂', '🌃', '🌄', '🌅', '🌆', '🌈', '🌉', '🌊', '🌋', '🌌', '🌍', '🌎', '🌏', '🌐', '🌑', '🌒', '🌓', '🌕', '🌗', '🌙', '🌛', '🌜', '🌞', '🌟', '🌠', '🌡️', '🌤️', '🌥️', '🌦️', '🌧️', '🌨️', '🌩️', '🌪', '🌪️', '🌫️', '🌬', '🌬️', '🌭', '🌮', '🌰', '🌱', '🌲', '🌳', '🌴', '🌵', '🌶️', '🌷', '🌸', '🌹', '🌺', '🌻', '🌼', '🌽', '🌾', '🌿', '🍀', '🍁', '🍂', '🍃', '🍄', '🍅', '🍆', '🍇', '🍈', '🍉', '🍊', '🍋', '🍌', '🍍', '🍎', '🍏', '🍐', '🍑', '🍒', '🍓', '🍔', '🍕', '🍖', '🍗', '🍘', '🍙', '🍚', '🍛', '🍜', '🍝', '🍞', '🍟', '🍠', '🍡', '🍢', '🍣', '🍤', '🍥', '🍦', '🍧', '🍨', '🍩', '🍪', '🍫', '🍬', '🍭', '🍮', '🍯', '🍰', '🍱', '🍲', '🍳', '🍴', '🍵', '🍶', '🍷', '🍸', '🍹', '🍺', '🍻', '🍼', '🍽', '🍽️', '🍾', '🍿', '🎀', '🎁', '🎂', '🎃', '🎄', '🎅', '🎇', '🎈', '🎉', '🎊', '🎋', '🎌', '🎎', '🎏', '🎒', '🎓', '🎖️', '🎙️', '🎚️', '🎛️', '🎟️', '🎠', '🎡', '🎢', '🎣', '🎤', '🎥', '🎧', '🎨', '🎩', '🎪', '🎬', '🎭', '🎮', '🎯', '🎱', '🎲', '🎳', '🎴', '🎵', '🎶', '🎸', '🎹', '🎺', '🎻', '🎼', '🎽', '🎾', '🎿', '🏀', '🏁', '🏂', '🏃', '🏃♀️', '🏃♂️', '🏄', '🏄♀️', '🏄♂️', '🏅', '🏆', '🏇', '🏊', '🏊♀️', '🏊♂️', '🏋️', '🏋️♀️', '🏋️♂️', '🏌️', '🏌️♀️', '🏌️♂️', '🏍️', '🏎️', '🏏', '🏐', '🏒', '🏓', '🏔️', '🏕', '🏕️', '🏖️', '🏗️', '🏙️', '🏚️', '🏛️', '🏜️', '🏝️', '🏞️', '🏟️', '🏠', '🏡', '🏢', '🏥', '🏦', '🏨', '🏪', '🏫', '🏬', '🏭', '🏮', '🏯', '🏰', '🏳️🌈', '🏴☠️', '🏵️', '🏷', '🏷️', '🏸', '🏹', '🏺', '🐂', '🐄', '🐅', '🐆', '🐇', '🐈', '🐉', '🐊', '🐋', '🐌', '🐍', '🐎', '🐐', '🐑', '🐒', '🐓', '🐔', '🐕', '🐕🦺', '🐖', '🐘', '🐙', '🐚', '🐛', '🐜', '🐝', '🐞', '🐟', '🐠', '🐡', '🐢', '🐣', '🐤', '🐥', '🐦', '🐧', '🐨', '🐩', '🐫', '🐬', '🐭', '🐮', '🐯', '🐰', '🐱', '🐲', '🐳', '🐴', '🐶', '🐷', '🐸', '🐹', '🐺', '🐻', '🐼', '🐾', '🐿️', '👀', '👁️', '👁️🗨️', '👂', '👃', '👇', '👈', '👉', '👊', '👋', '👌', '👌🏽', '👍', '👍🏻', '👍🏼', '👍🏽', '👍🏿', '👎', '👏', '👐', '👑', '👒', '👓', '👔', '👕', '👖', '👗', '👙', '👚', '👛', '👜', '👝', '👞', '👟', '👠', '👡', '👢', '👣', '👤', '👥', '👦', '👦🏼', '👧', '👨', '👨⚕️', '👨✈️', '👨🍳', '👨🎓', '👨🎤', '👨🏫', '👨👦', '👨👧👦', '👨👩👦', '👨👩👦👦', '👨👩👧', '👨👩👧👦', '👨💻', '👨💼', '👨🔧', '👨🚀', '👨🚒', '👨🦰', '👨', '👩', '👩⚕️', '👩✈️', '👩❤️👨', '👩🍳', '👩🍼', '👩🎓', '👩🎤', '👩🎨', '👩🏫', '👩👦', '👩👧', '👩👧👦', '👩💻', '👩💼', '👩🔧', '👩🔬', '👩🦰', '👩🦳', '👪', '👫', '👬', '👭', '👮♂️', '👯♂️', '👰', '👰♀️', '👴', '👵', '👶', '👶🏻', '👷♀️', '👷♂️', '👸', '👹', '👻', '👼', '👽', '👾', '💀', '💁♀️', '💁♂️', '💃', '💃🏻', '💃🏽', '💄', '💅', '💆', '💆♀️', '💆♂️', '💇♀️', '💇♂️', '💈', '💉', '💊', '💋', '💌', '💍', '💎', '💏', '💐', '💑', '💒', '💓', '💔', '💕', '💖', '💗', '💘', '💙', '💚', '💛', '💜', '💝', '💞', '💡', '💢', '💤', '💥', '💦', '💧', '💨', '💩', '💪', '💪🏻', '💪🏼', '💪🏽', '💪🏿', '💫', '💬', '💭', '💯', '💰', '💳', '💵', '💸', '💺', '💻', '💼', '💽', '💾', '📀', '📁', '📂', '📄', '📅', '📆', '📇', '📈', '📉', '📊', '📋', '📌', '📍', '📎', '📏', '📐', '📑', '📒', '📓', '📖', '📘', '📚', '📜', '📝', '📞', '📠', '📡', '📢', '📣', '📥', '📦', '📧', '📨', '📩', '📬', '📰', '📱', '📲', '📵', '📶', '📷', '📸', '📹', '📺', '📻', '📼', '📽️', '📿', '🔁', '🔄', '🔆', '🔇', '🔊', '🔋', '🔌', '🔍', '🔎', '🔐', '🔑', '🔒', '🔓', '🔔', '🔕', '🔖', '🔗', '🔘', '🔝', '🔢', '🔤', '🔥', '🔦', '🔧', '🔨', '🔩', '🔪', '🔫', '🔬', '🔭', '🔮', '🔴', '🔵', '🔶', '🔷', '🔸', '🔹', '🔺', '🕉️', '🕊️', '🕌', '🕐', '🕑', '🕒', '🕔', '🕖', '🕗', '🕘', '🕛', '🕯️', '🕰', '🕰️', '🕴', '🕴️', '🕵️', '🕵️♀️', '🕵️♂️', '🕶️', '🕷️', '🕸️', '🕹️', '🕺', '🕺🏽', '🖇️', '🖊️', '🖋️', '🖌️', '🖍️', '🖐️', '🖤', '🖥️', '🖨️', '🖱️', '🖼️', '🗂️', '🗃️', '🗄️', '🗑️', '🗒️', '🗓️', '🗝️', '🗡️', '🗣️', '🗨️', '🗳️', '🗺️', '🗼', '🗽', '😁', '😂', '😃', '😄', '😅', '😇', '😈', '😉', '😊', '😋', '😌', '😍', '😎', '😏', '😐', '😒', '😓', '😔', '😕', '😖', '😗', '😘', '😞', '😟', '😠', '😡', '😢', '😣', '😤', '😥', '😧', '😨', '😩', '😫', '😬', '😮', '😮💨', '😯', '😰', '😱', '😲', '😳', '😴', '😶', '😷', '😸', '😺', '😻', '😼', '😿', '🙁', '🙄', '🙅♂️', '🙇♂️', '🙈', '🙋♀️', '🙋♂️', '🙌', '🙌🏽', '🙏', '🚀', '🚁', '🚂', '🚄', '🚆', '🚉', '🚌', '🚍', '🚐', '🚑', '🚒', '🚓', '🚔', '🚗', '🚘', '🚙', '🚚', '🚛', '🚜', '🚢', '🚣♀️', '🚣♂️', '🚤', '🚦', '🚧', '🚨', '🚩', '🚪', '🚫', '🚬', '🚭', '🚮', '🚰', '🚱', '🚲', '🚴', '🚴♀️', '🚴♂️', '🚵♀️', '🚵♂️', '🚶', '🚶♀️', '🚶♂️', '🚸', '🚺', '🚼', '🚽', '🚿', '🛀', '🛁', '🛋️', '🛌', '🛍', '🛍️', '🛎️', '🛏️', '🛑', '🛒', '🛝', '🛟', '🛠', '🛠️', '🛡', '🛡️', '🛢️', '🛣️', '🛤️', '🛥️', '🛩️', '🛫', '🛬', '🛳️', '🛴', '🛵', '🛶', '🛸', '🛹', '🛻', '🛼', '🟡', '🟢', '🟣', '🟥', '🟦', '🟩', '🤍', '🤎', '🤐', '🤑', '🤒', '🤓', '🤔', '🤕', '🤖', '🤗', '🤘', '🤛', '🤝', '🤠', '🤢', '🤣', '🤤', '🤦♀️', '🤦♂️', '🤧', '🤨', '🤩', '🤪', '🤫', '🤭', '🤯', '🤰', '🤱', '🤲', '🤳', '🤵', '🤷♀️', '🤷♂️', '🤸', '🤸♀️', '🤸♂️', '🤹♂️', '🤼♂️', '🤾♂️', '🤿', '🥀', '🥁', '🥂', '🥃', '🥄', '🥅', '🥇', '🥉', '🥊', '🥋', '🥐', '🥑', '🥒', '🥓', '🥔', '🥕', '🥖', '🥗', '🥘', '🥙', '🥚', '🥛', '🥜', '🥝', '🥞', '🥟', '🥡', '🥢', '🥣', '🥤', '🥥', '🥦', '🥧', '🥩', '🥪', '🥫', '🥬', '🥭', '🥮', '🥯', '🥰', '🥱', '🥲', '🥳', '🥴', '🥵', '🥶', '🥷', '🥹', '🥺', '🥾', '🥿', '🦀', '🦁', '🦄', '🦅', '🦆', '🦇', '🦈', '🦉', '🦊', '🦋', '🦌', '🦎', '🦏', '🦐', '🦑', '🦒', '🦓', '🦔', '🦕', '🦖', '🦗', '🦙', '🦚', '🦜', '🦞', '🦟', '🦠', '🦢', '🦣', '🦩', '🦪', '🦮', '🦴', '🦵', '🦶', '🦷', '🦸♀️', '🦸♂️', '🦺', '🦻', '🦾', '🧀', '🧁', '🧂', '🧃', '🧄', '🧇', '🧈', '🧉', '🧊', '🧋', '🧐', '🧑⚕️', '🧑✈️', '🧑🌾', '🧑🍳', '🧑🎓', '🧑🎤', '🧑🏭', '🧑💻', '🧑💼', '🧑🚀', '🧑🚒', '🧑🤝🧑', '🧑🦰', '🧒', '🧒🏻', '🧒🏽', '🧓', '🧔', '🧖♀️', '🧖♂️', '🧗♂', '🧗♂️', '🧘', '🧘♀️', '🧘♂️', '🧙♀️', '🧙♂️', '🧚', '🧚♀️', '🧚♂️', '🧜♀️', '🧜♂️', '🧝', '🧟♂️', '🧠', '🧡', '🧢', '🧣', '🧤', '🧥', '🧦', '🧧', '🧩', '🧪', '🧬', '🧭', '🧮', '🧰', '🧱', '🧲', '🧳', '🧴', '🧵', '🧶', '🧸', '🧹', '🧺', '🧻', '🧼', '🧽', '🧾', '🧿', '🩰', '🩱', '🩲', '🩳', '🩴', '🩷', '🩸', '🩹', '🩺', '🪁', '🪄', '🪐', '🪑', '🪒', '🪓', '🪖', '🪘', '🪙', '🪚', '🪞', '🪟', '🪥', '🪨', '🪳', '🪴', '🪵', '🫀', '🫁', '🫐', '🫖')
additional_specific_tokens = ['~~~', '---', '**']

tokens_to_potentially_add = list(set(additional_specific_tokens + list(VALID_ENDINGS)))

# Load tokenizer
print(f"Loading tokenizer for '{MODEL_NAME}'...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
original_vocab_size = len(tokenizer)
print(f"  Original vocab size: {original_vocab_size}")

# Pad token
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
    print(f"  Set pad_token = eos_token ({tokenizer.eos_token})")
tokenizer.padding_side = "right"  # Right pad for training

# Verify Qwen special tokens exist
im_start_id = tokenizer.convert_tokens_to_ids("<|im_start|>")
im_end_id = tokenizer.convert_tokens_to_ids("<|im_end|>")
if isinstance(im_start_id, int) and im_start_id != tokenizer.unk_token_id:
    print(f"  <|im_start|> ID: {im_start_id}, <|im_end|> ID: {im_end_id}")
    tokens_to_potentially_add = [t for t in tokens_to_potentially_add
                                  if t not in ["<|im_start|>", "<|im_end|>"]]

# Add custom tokens
num_added = tokenizer.add_tokens(tokens_to_potentially_add, special_tokens=False)
print(f"  Added {num_added} new tokens → vocab size: {len(tokenizer)}")

# Save tokenizer
tokenizer.save_pretrained(TOKENIZER_SAVE_DIR)
print(f"  Tokenizer saved to: {TOKENIZER_SAVE_DIR}")
print("CELL 3 DONE\n")

CELL 3: TOKENIZER PREPARATION
Loading tokenizer for 'Qwen/Qwen3-0.6B'...


config.json:   0%|          | 0.00/726 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

  Original vocab size: 151669
  <|im_start|> ID: 151644, <|im_end|> ID: 151645
  Added 1083 new tokens → vocab size: 152746
  Tokenizer saved to: /kaggle/working/output/tokenizer
CELL 3 DONE



## Cell 4: Data Processing & Tokenization

In [5]:
print("=" * 60)
print("CELL 4: DATA PROCESSING & TOKENIZATION")
print("=" * 60)

# ---- Step 1: Copy CSVs to local working dir for speed ----
print("Copying CSV files to local storage...")
csv_map = {
    'train': TRAIN_CSV_PATH,
    'val':   VAL_CSV_PATH,
    'test':  TEST_CSV_PATH,
}
for split_name, src_path in csv_map.items():
    dst_path = os.path.join(LOCAL_DATA_DIR, f'{split_name}.csv')
    if not os.path.exists(dst_path):
        if os.path.exists(src_path):
            shutil.copy2(src_path, dst_path)
            print(f"  Copied {split_name}.csv to local")
        else:
            print(f"  ⚠ WARNING: {src_path} not found! Make sure KAGGLE_INPUT_DIR is correct.")
    else:
        print(f"  {split_name}.csv already local")


# ---- Step 2: Process CSV → ChatML ----
def process_csv_to_chatml(csv_path, split_name):
    """Read CSV, create ChatML prompt / full_text / completion columns."""
    df = pd.read_csv(csv_path)
    print(f"  [{split_name}] Loaded {len(df)} rows from CSV")

    if 'id' in df.columns:
        df.drop_duplicates(subset=['id'], keep='first', inplace=True)
        print(f"  [{split_name}] After dedup: {len(df)} rows")

    pn_col = 'product_name' if 'product_name' in df.columns else None
    desc_col = 'cleaned_description' if 'cleaned_description' in df.columns else (
        'description' if 'description' in df.columns else None
    )
    adv_col = 'advertisement' if 'advertisement' in df.columns else None
    if adv_col is None:
        raise ValueError(f"Column 'advertisement' not found in {csv_path}")

    records = []
    for _, row in df.iterrows():
        name = str(row.get(pn_col, "")).strip() if pn_col else ""
        desc = str(row.get(desc_col, "")).strip() if desc_col else ""
        adv  = str(row.get(adv_col, "")).strip()
        if not adv:
            continue

        user_content = f"tạo quảng cáo cho sản phẩm sau:\nTên sản phẩm: {name}\nMô tả: {desc}"
        prompt    = f"<|im_start|>user\n{user_content}<|im_end|>\n<|im_start|>assistant\n"
        full_text = f"{prompt}{adv}<|im_end|>"
        records.append({
            'prompt': prompt,
            'completion': adv,
            'full_text': full_text,
        })

    df_out = pd.DataFrame(records)
    print(f"  [{split_name}] After processing: {len(df_out)} valid samples")
    return df_out


# ---- Step 3: Tokenize with label masking (NO padding) ----
def tokenize_for_training(examples):
    """
    Batch tokenization. No padding — the collator handles per-batch padding.
    Labels: -100 for prompt portion, real IDs for completion.
    """
    full_enc = tokenizer(
        examples["full_text"],
        max_length=MAX_SEQ_LENGTH,
        truncation=True,
        padding=False,
        add_special_tokens=False,
    )
    prompt_enc = tokenizer(
        examples["prompt"],
        max_length=MAX_SEQ_LENGTH,
        truncation=True,
        padding=False,
        add_special_tokens=False,
    )

    all_labels = []
    for i in range(len(full_enc["input_ids"])):
        ids = full_enc["input_ids"][i]
        prompt_len = min(len(prompt_enc["input_ids"][i]), len(ids))
        labels = [-100] * prompt_len + list(ids[prompt_len:])
        all_labels.append(labels)

    return {
        "input_ids":      full_enc["input_ids"],
        "attention_mask":  full_enc["attention_mask"],
        "labels":          all_labels,
    }


# ---- Step 4: Process each split ----
datasets = {}
for split_name in ['train', 'val', 'test']:
    local_csv = os.path.join(LOCAL_DATA_DIR, f'{split_name}.csv')
    cache_path = os.path.join(LOCAL_DATA_DIR, f'{split_name}_tokenized')

    if not os.path.exists(local_csv):
        print(f"  ⚠ {split_name}.csv not found, skipping")
        datasets[split_name] = None
        continue

    if os.path.isdir(cache_path):
        print(f"  Loading cached tokenized {split_name} dataset...")
        datasets[split_name] = load_from_disk(cache_path)
        print(f"  [{split_name}] {len(datasets[split_name])} samples loaded from cache")
    else:
        df = process_csv_to_chatml(local_csv, split_name)
        if df.empty:
            print(f"  ⚠ {split_name} is empty!")
            datasets[split_name] = None
            continue

        hf_ds = Dataset.from_pandas(df[['prompt', 'full_text']])
        print(f"  [{split_name}] Tokenizing {len(hf_ds)} samples...")
        tok_ds = hf_ds.map(
            tokenize_for_training,
            batched=True,
            batch_size=1000,
            remove_columns=['prompt', 'full_text'],
            desc=f"Tokenizing {split_name}",
        )
        tok_ds.save_to_disk(cache_path)
        datasets[split_name] = tok_ds
        print(f"  [{split_name}] Tokenized & cached: {len(tok_ds)} samples")

    # Print length statistics
    ds = datasets[split_name]
    if ds is not None and len(ds) > 0:
        lengths = [len(x) for x in ds['input_ids']]
        print(f"  [{split_name}] Seq lengths — min: {min(lengths)}, max: {max(lengths)}, "
              f"mean: {np.mean(lengths):.0f}, median: {np.median(lengths):.0f}")

train_dataset = datasets.get('train')
eval_dataset  = datasets.get('val')
test_dataset  = datasets.get('test')

if train_dataset is None or len(train_dataset) == 0:
    raise RuntimeError("Train dataset is empty. Check KAGGLE_INPUT_DIR and CSV file names.")
print(f"\nDataset sizes — Train: {len(train_dataset)}, "
      f"Val: {len(eval_dataset) if eval_dataset else 0}, "
      f"Test: {len(test_dataset) if test_dataset else 0}")
print("CELL 4 DONE\n")

CELL 4: DATA PROCESSING & TOKENIZATION
Copying CSV files to local storage...
  Copied train.csv to local
  Copied val.csv to local
  Copied test.csv to local
  [train] Loaded 80125 rows from CSV
  [train] After dedup: 80125 rows
  [train] After processing: 80125 valid samples
  [train] Tokenizing 80125 samples...


Tokenizing train:   0%|          | 0/80125 [00:00<?, ? examples/s]

Saving the dataset (0/3 shards):   0%|          | 0/80125 [00:00<?, ? examples/s]

  [train] Tokenized & cached: 80125 samples
  [train] Seq lengths — min: 70, max: 2048, mean: 1328, median: 1286
  [val] Loaded 4452 rows from CSV
  [val] After dedup: 4452 rows
  [val] After processing: 4452 valid samples
  [val] Tokenizing 4452 samples...


Tokenizing val:   0%|          | 0/4452 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/4452 [00:00<?, ? examples/s]

  [val] Tokenized & cached: 4452 samples
  [val] Seq lengths — min: 130, max: 2048, mean: 1329, median: 1287
  [test] Loaded 4452 rows from CSV
  [test] After dedup: 4452 rows
  [test] After processing: 4452 valid samples
  [test] Tokenizing 4452 samples...


Tokenizing test:   0%|          | 0/4452 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/4452 [00:00<?, ? examples/s]

  [test] Tokenized & cached: 4452 samples
  [test] Seq lengths — min: 161, max: 2048, mean: 1327, median: 1284

Dataset sizes — Train: 80125, Val: 4452, Test: 4452
CELL 4 DONE



## Cell 5: Dynamic Padding Data Collator

In [6]:
print("=" * 60)
print("CELL 5: DATA COLLATOR")
print("=" * 60)

class DynamicPaddingDataCollator:
    """
    Pads to the max length within each batch (NOT the global MAX_SEQ_LENGTH).
    This saves enormous compute when sequences vary in length.
    """
    def __init__(self, tokenizer, max_length=None):
        self.pad_token_id = tokenizer.pad_token_id
        self.max_length = max_length

    def __call__(self, features):
        batch_max_len = max(len(f['input_ids']) for f in features)
        if self.max_length:
            batch_max_len = min(batch_max_len, self.max_length)

        batch_input_ids = []
        batch_attention_mask = []
        batch_labels = []

        for f in features:
            ids    = list(f['input_ids'][:batch_max_len])
            mask   = list(f['attention_mask'][:batch_max_len])
            labels = list(f['labels'][:batch_max_len])
            pad_len = batch_max_len - len(ids)

            # Right padding
            batch_input_ids.append(ids + [self.pad_token_id] * pad_len)
            batch_attention_mask.append(mask + [0] * pad_len)
            batch_labels.append(labels + [-100] * pad_len)

        return {
            'input_ids':      torch.tensor(batch_input_ids, dtype=torch.long),
            'attention_mask':  torch.tensor(batch_attention_mask, dtype=torch.long),
            'labels':          torch.tensor(batch_labels, dtype=torch.long),
        }

data_collator = DynamicPaddingDataCollator(tokenizer, max_length=MAX_SEQ_LENGTH)
print("DynamicPaddingDataCollator initialized.")
print("CELL 5 DONE\n")

CELL 5: DATA COLLATOR
DynamicPaddingDataCollator initialized.
CELL 5 DONE



## Cell 6: Model + LoRA Setup

In [7]:
print("=" * 60)
print("CELL 6: MODEL + LoRA SETUP")
print("=" * 60)

print(f"Loading base model '{MODEL_NAME}' in {COMPUTE_DTYPE}...")
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=COMPUTE_DTYPE,
    trust_remote_code=True,
    device_map=None,  # Let Trainer handle multi-GPU placement
)
print(f"  Base model loaded. Params: {sum(p.numel() for p in model.parameters()):,}")

# Resize embeddings if we added tokens
if model.get_input_embeddings().weight.shape[0] != len(tokenizer):
    old_size = model.get_input_embeddings().weight.shape[0]
    model.resize_token_embeddings(len(tokenizer))
    print(f"  Resized embeddings: {old_size} → {len(tokenizer)}")

# Apply LoRA
print("Applying LoRA...")
peft_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    inference_mode=False,
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    target_modules=LORA_TARGET_MODULES,
    bias="none",
)
model = get_peft_model(model, peft_config)
model.print_trainable_parameters()

# Enable gradient checkpointing for larger batch sizes on T4's 16GB
model.enable_input_require_grads()

# VRAM report (per GPU)
for i in range(n_gpus):
    total_memory = torch.cuda.get_device_properties(i).total_memory / 1024**3
    print(f"  GPU {i}: {total_memory:.1f} GiB total VRAM")

print("CELL 6 DONE\n")

CELL 6: MODEL + LoRA SETUP
Loading base model 'Qwen/Qwen3-0.6B' in torch.float16...


`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors:   0%|          | 0.00/1.50G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/311 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie model.embed_tokens.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


generation_config.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

  Base model loaded. Params: 751,632,384


The new embeddings will be initialized from a multivariate normal distribution that has old embeddings' mean and covariance. As described in this article: https://nlp.stanford.edu/~johnhew/vocab-expansion.html. To disable this, use `mean_resizing=False`
The new lm_head weights will be initialized from a multivariate normal distribution that has old embeddings' mean and covariance. As described in this article: https://nlp.stanford.edu/~johnhew/vocab-expansion.html. To disable this, use `mean_resizing=False`


  Resized embeddings: 151936 → 152746
Applying LoRA...
trainable params: 5,046,272 || all params: 601,925,632 || trainable%: 0.8384
  GPU 0: 14.6 GiB total VRAM
  GPU 1: 14.6 GiB total VRAM
CELL 6 DONE



## Cell 7: Training

In [8]:
# print("=" * 60)
# print("CELL 7: TRAINING")
# print("=" * 60)

# import inspect
# import transformers as _tf
# print(f"  Transformers version: {_tf.__version__}")

# # Calculate expected training metrics
# n_train = len(train_dataset)
# eff_batch = PER_DEVICE_TRAIN_BATCH_SIZE * max(n_gpus, 1) * GRADIENT_ACCUMULATION_STEPS
# steps_per_epoch = (n_train + eff_batch - 1) // eff_batch
# total_steps = steps_per_epoch * NUM_TRAIN_EPOCHS
# warmup_steps = int(total_steps * WARMUP_RATIO)
# logging_steps = max(1, steps_per_epoch // 20)

# print(f"  Training samples:     {n_train}")
# print(f"  Effective batch size: {eff_batch}  ({PER_DEVICE_TRAIN_BATCH_SIZE}/GPU × {n_gpus} GPUs × {GRADIENT_ACCUMULATION_STEPS} accum)")
# print(f"  Steps per epoch:      {steps_per_epoch}")
# print(f"  Total steps:          {total_steps}")
# print(f"  Warmup steps:         {warmup_steps}")
# print(f"  Logging every:        {logging_steps} steps")

# # ---- Build TrainingArguments (auto-filter unsupported params) ----
# _valid_ta_args = set(inspect.signature(TrainingArguments.__init__).parameters.keys())

# training_kwargs = dict(
#     output_dir=CHECKPOINT_DIR,
#     num_train_epochs=NUM_TRAIN_EPOCHS,
#     per_device_train_batch_size=PER_DEVICE_TRAIN_BATCH_SIZE,
#     per_device_eval_batch_size=PER_DEVICE_EVAL_BATCH_SIZE,
#     gradient_accumulation_steps=GRADIENT_ACCUMULATION_STEPS,

#     # Optimizer
#     learning_rate=LEARNING_RATE,
#     weight_decay=WEIGHT_DECAY,
#     warmup_steps=warmup_steps,
#     lr_scheduler_type="cosine",
#     optim=OPTIM,

#     # Precision — FP16 for T4 (no BF16 support)
#     fp16=True,
#     bf16=False,

#     # Checkpointing
#     save_strategy="epoch",
#     save_total_limit=3,
#     load_best_model_at_end=True,
#     metric_for_best_model="eval_loss",
#     greater_is_better=False,

#     # Performance
#     gradient_checkpointing=True,
#     dataloader_num_workers=2,
#     dataloader_pin_memory=True,

#     # Logging
#     logging_strategy="steps",
#     logging_steps=logging_steps,
#     report_to=["tensorboard"],

#     # Misc
#     remove_unused_columns=True,
#     seed=42,
# )

# # Handle params that were renamed or added across versions
# if "eval_strategy" in _valid_ta_args:
#     training_kwargs["eval_strategy"] = "epoch"
# elif "evaluation_strategy" in _valid_ta_args:
#     training_kwargs["evaluation_strategy"] = "epoch"

# if "gradient_checkpointing_kwargs" in _valid_ta_args:
#     training_kwargs["gradient_checkpointing_kwargs"] = {"use_reentrant": False}

# if "group_by_length" in _valid_ta_args:
#     training_kwargs["group_by_length"] = True

# # Auto-remove any params not supported by this transformers version
# unsupported_ta = [k for k in training_kwargs if k not in _valid_ta_args]
# for k in unsupported_ta:
#     print(f"  ⚠ Removing unsupported TrainingArguments param: '{k}'")
#     del training_kwargs[k]

# training_args = TrainingArguments(**training_kwargs)
# print("TrainingArguments initialized.")

# # ---- Build Trainer (handle v4 'tokenizer' vs v5 'processing_class') ----
# _valid_trainer_args = set(inspect.signature(Trainer.__init__).parameters.keys())

# trainer_kwargs = dict(
#     model=model,
#     args=training_args,
#     train_dataset=train_dataset,
#     eval_dataset=eval_dataset,
#     data_collator=data_collator,
# )

# if "processing_class" in _valid_trainer_args:
#     trainer_kwargs["processing_class"] = tokenizer    # transformers v5+
# elif "tokenizer" in _valid_trainer_args:
#     trainer_kwargs["tokenizer"] = tokenizer            # transformers v4

# trainer = Trainer(**trainer_kwargs)
# print("Trainer initialized.")

# # ---- Train ----
# print("\nStarting training...")
# t_start = time.time()
# train_result = trainer.train()
# t_elapsed = time.time() - t_start

# print(f"\n✅ Training complete in {t_elapsed / 60:.1f} minutes")
# if hasattr(train_result, 'metrics'):
#     print("Training metrics:", train_result.metrics)

# # Save training metrics
# metrics_path = os.path.join(OUTPUT_DIR, "training_metrics.json")
# with open(metrics_path, "w") as f:
#     json.dump(train_result.metrics, f, indent=2)
# trainer.save_state()
# print("CELL 7 DONE\n")

In [ ]:
print("=" * 60)
print("CELL 7: TRAINING")
print("=" * 60)

import inspect
import transformers as _tf
print(f"  Transformers version: {_tf.__version__}")

# Calculate expected training metrics
n_train = len(train_dataset)
eff_batch = PER_DEVICE_TRAIN_BATCH_SIZE * max(n_gpus, 1) * GRADIENT_ACCUMULATION_STEPS
steps_per_epoch = (n_train + eff_batch - 1) // eff_batch
total_steps = steps_per_epoch * NUM_TRAIN_EPOCHS
warmup_steps = int(total_steps * WARMUP_RATIO)
logging_steps = max(1, steps_per_epoch // 20)

print(f"  Training samples:     {n_train}")
print(f"  Effective batch size: {eff_batch}  ({PER_DEVICE_TRAIN_BATCH_SIZE}/GPU × {n_gpus} GPUs × {GRADIENT_ACCUMULATION_STEPS} accum)")
print(f"  Steps per epoch:      {steps_per_epoch}")
print(f"  Total steps:          {total_steps}")
print(f"  Warmup steps:         {warmup_steps}")
print(f"  Logging every:        {logging_steps} steps")

# ---- Build TrainingArguments (auto-filter unsupported params) ----
_valid_ta_args = set(inspect.signature(TrainingArguments.__init__).parameters.keys())

training_kwargs = dict(
    output_dir=CHECKPOINT_DIR,
    num_train_epochs=NUM_TRAIN_EPOCHS,
    per_device_train_batch_size=PER_DEVICE_TRAIN_BATCH_SIZE,
    gradient_accumulation_steps=GRADIENT_ACCUMULATION_STEPS,

    # Optimizer
    learning_rate=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY,
    warmup_steps=warmup_steps,
    lr_scheduler_type="cosine",
    optim=OPTIM,

    # Precision — FP16 for T4 (no BF16 support)
    fp16=True,
    bf16=False,

    # No evaluation
    # eval_strategy / evaluation_strategy = "no" (handled below)

    # Checkpointing — save every epoch, keep all 5
    save_strategy="epoch",
    save_total_limit=NUM_TRAIN_EPOCHS,

    # Performance
    gradient_checkpointing=True,
    dataloader_num_workers=2,
    dataloader_pin_memory=True,

    # Logging
    logging_strategy="steps",
    logging_steps=logging_steps,
    report_to=["tensorboard"],

    # Misc
    remove_unused_columns=True,
    seed=42,
)

# Handle renamed params across versions
if "eval_strategy" in _valid_ta_args:
    training_kwargs["eval_strategy"] = "no"
elif "evaluation_strategy" in _valid_ta_args:
    training_kwargs["evaluation_strategy"] = "no"

if "gradient_checkpointing_kwargs" in _valid_ta_args:
    training_kwargs["gradient_checkpointing_kwargs"] = {"use_reentrant": False}

if "group_by_length" in _valid_ta_args:
    training_kwargs["group_by_length"] = True

# Auto-remove any params not supported by this transformers version
unsupported_ta = [k for k in training_kwargs if k not in _valid_ta_args]
for k in unsupported_ta:
    print(f"  ⚠ Removing unsupported TrainingArguments param: '{k}'")
    del training_kwargs[k]

training_args = TrainingArguments(**training_kwargs)
print("TrainingArguments initialized (no eval during training).")

# ---- Build Trainer (handle v4 'tokenizer' vs v5 'processing_class') ----
_valid_trainer_args = set(inspect.signature(Trainer.__init__).parameters.keys())

trainer_kwargs = dict(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    data_collator=data_collator,
)

if "processing_class" in _valid_trainer_args:
    trainer_kwargs["processing_class"] = tokenizer
elif "tokenizer" in _valid_trainer_args:
    trainer_kwargs["tokenizer"] = tokenizer

trainer = Trainer(**trainer_kwargs)
print("Trainer initialized.")

# ---- Train ----
print("\nStarting training...")
t_start = time.time()
train_result = trainer.train()
t_elapsed = time.time() - t_start

print(f"\n✅ Training complete in {t_elapsed / 60:.1f} minutes")
if hasattr(train_result, 'metrics'):
    print("Training metrics:", train_result.metrics)

# Save training metrics
metrics_path = os.path.join(OUTPUT_DIR, "training_metrics.json")
with open(metrics_path, "w") as f:
    json.dump(train_result.metrics, f, indent=2)
trainer.save_state()
print("CELL 7 DONE\n")

CELL 7: TRAINING
  Transformers version: 5.2.0
  Training samples:     80125
  Effective batch size: 32  (2/GPU × 2 GPUs × 8 accum)
  Steps per epoch:      2504
  Total steps:          1252.0
  Warmup steps:         37
  Logging every:        125 steps
TrainingArguments initialized (no eval during training).


2026-02-18 11:40:58.901385: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1771414859.330301     113 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1771414859.464800     113 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1771414860.403609     113 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1771414860.403636     113 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1771414860.403674     113 computation_placer.cc:177] computation placer alr

Trainer initialized.

Starting training...


huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


Step,Training Loss
125,1.394501
250,1.167567
375,1.112432


## Cell 8: Save Final Model

In [ ]:
print("=" * 60)
print("CELL 8: SAVE FINAL MODEL")
print("=" * 60)

print(f"Saving best model adapter to: {FINAL_MODEL_DIR}")
trainer.save_model(FINAL_MODEL_DIR)
tokenizer.save_pretrained(FINAL_MODEL_DIR)
shutil.copy2(metrics_path, os.path.join(FINAL_MODEL_DIR, "training_metrics.json"))

trainer_state_path = os.path.join(CHECKPOINT_DIR, "trainer_state.json")
if os.path.exists(trainer_state_path):
    shutil.copy2(trainer_state_path, os.path.join(FINAL_MODEL_DIR, "trainer_state.json"))

print(f"✅ Model, tokenizer, and metrics saved to: {FINAL_MODEL_DIR}")
print("Files:")
for f in os.listdir(FINAL_MODEL_DIR):
    fpath = os.path.join(FINAL_MODEL_DIR, f)
    size_kb = os.path.getsize(fpath) / 1024
    print(f"  {f} ({size_kb:.1f} KB)")
# Copy checkpoint cuối cùng (có optimizer state) ra output để resume được
import glob
ckpt_dirs = sorted(glob.glob(os.path.join(CHECKPOINT_DIR, "checkpoint-*")))
if ckpt_dirs:
    last_ckpt = ckpt_dirs[-1]
    resume_dir = os.path.join(OUTPUT_DIR, "last_checkpoint_for_resume")
    if os.path.exists(resume_dir):
        shutil.rmtree(resume_dir)
    shutil.copytree(last_ckpt, resume_dir)
    print(f"✅ Full checkpoint (with optimizer) copied to: {resume_dir}")
print("CELL 8 DONE\n")

## Cell 9: Post-Training Evaluation (BLEU / ROUGE)

In [ ]:
# print("=" * 60)
# print("CELL 9: POST-TRAINING EVALUATION")
# print("=" * 60)

# def run_generation_evaluation(model, tokenizer, csv_path, split_name, max_samples=None,
#                               max_new_tokens=512, num_beams=4, batch_size=8):
#     """
#     Generate predictions and compute BLEU / ROUGE.
#     Uses left-padding for correct batch generation.
#     """
#     df = process_csv_to_chatml(csv_path, split_name)
#     if max_samples and len(df) > max_samples:
#         df = df.head(max_samples)
#         print(f"  Evaluating on first {max_samples} samples")

#     prompts = df['prompt'].tolist()
#     references = df['completion'].tolist()

#     # Switch to left padding for generation
#     original_padding_side = tokenizer.padding_side
#     tokenizer.padding_side = "left"

#     model.eval()
#     # Move to single GPU for generation (avoid DataParallel issues)
#     device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
#     # If model is wrapped in DataParallel, unwrap it
#     gen_model = model.module if hasattr(model, 'module') else model
#     gen_model = gen_model.to(device)

#     all_predictions = []

#     eos_ids = [tokenizer.eos_token_id]
#     im_end_id_val = tokenizer.convert_tokens_to_ids("<|im_end|>")
#     if im_end_id_val != tokenizer.unk_token_id and im_end_id_val not in eos_ids:
#         eos_ids.append(im_end_id_val)

#     for i in tqdm(range(0, len(prompts), batch_size), desc=f"Generating ({split_name})"):
#         batch_prompts = prompts[i:i + batch_size]
#         inputs = tokenizer(
#             batch_prompts,
#             return_tensors="pt",
#             padding=True,
#             truncation=True,
#             max_length=MAX_SEQ_LENGTH - max_new_tokens,
#             add_special_tokens=False,
#         ).to(device)

#         with torch.no_grad():
#             output_ids = gen_model.generate(
#                 **inputs,
#                 max_new_tokens=max_new_tokens,
#                 num_beams=num_beams,
#                 early_stopping=True if num_beams > 1 else False,
#                 no_repeat_ngram_size=3,
#                 pad_token_id=tokenizer.pad_token_id,
#                 eos_token_id=eos_ids,
#             )

#         for j in range(len(batch_prompts)):
#             input_len = inputs['attention_mask'][j].sum().item()
#             gen_ids = output_ids[j][input_len:]
#             text = tokenizer.decode(gen_ids, skip_special_tokens=True).strip()
#             all_predictions.append(text if text else "EMPTY_PRED")

#     # Restore padding side
#     tokenizer.padding_side = original_padding_side

#     # Compute metrics
#     metric_bleu = evaluate.load("bleu")
#     metric_rouge = evaluate.load("rouge")
#     results = {}

#     try:
#         bleu_refs = [[ref] for ref in references]
#         bleu = metric_bleu.compute(predictions=all_predictions, references=bleu_refs)
#         results['bleu'] = round(bleu['bleu'], 4)
#     except Exception as e:
#         print(f"  BLEU error: {e}")
#         results['bleu'] = 0.0

#     try:
#         rouge = metric_rouge.compute(
#             predictions=all_predictions, references=references,
#             use_stemmer=False, rouge_types=["rouge1", "rouge2", "rougeLsum"],
#         )
#         results['rouge1']    = round(rouge['rouge1'], 4)
#         results['rouge2']    = round(rouge['rouge2'], 4)
#         results['rougeLsum'] = round(rouge['rougeLsum'], 4)
#     except Exception as e:
#         print(f"  ROUGE error: {e}")
#         results['rouge1'] = results['rouge2'] = results['rougeLsum'] = 0.0

#     print(f"\n  [{split_name}] Metrics: {results}")
#     print(f"\n  --- Sample predictions ({split_name}) ---")
#     for idx in range(min(3, len(all_predictions))):
#         print(f"  [Sample {idx+1}]")
#         print(f"    Reference:  {references[idx][:200]}...")
#         print(f"    Prediction: {all_predictions[idx][:200]}...")
#         print()

#     return results


# # Evaluate on validation set
# if eval_dataset is not None and len(eval_dataset) > 0:
#     val_csv_local = os.path.join(LOCAL_DATA_DIR, 'val.csv')
#     if os.path.exists(val_csv_local):
#         print("Evaluating on VALIDATION set...")
#         val_metrics = run_generation_evaluation(
#             model=trainer.model, tokenizer=tokenizer,
#             csv_path=val_csv_local, split_name="val",
#             max_new_tokens=512, num_beams=4, batch_size=8,
#         )
#         val_metrics_path = os.path.join(FINAL_MODEL_DIR, "val_generation_metrics.json")
#         with open(val_metrics_path, "w") as f:
#             json.dump(val_metrics, f, indent=2)
#         print(f"  Saved to: {val_metrics_path}")

# # Evaluate on test set
# if test_dataset is not None and len(test_dataset) > 0:
#     test_csv_local = os.path.join(LOCAL_DATA_DIR, 'test.csv')
#     if os.path.exists(test_csv_local):
#         print("\nEvaluating on TEST set...")
#         test_metrics = run_generation_evaluation(
#             model=trainer.model, tokenizer=tokenizer,
#             csv_path=test_csv_local, split_name="test",
#             max_new_tokens=512, num_beams=4, batch_size=8,
#         )
#         test_metrics_path = os.path.join(FINAL_MODEL_DIR, "test_generation_metrics.json")
#         with open(test_metrics_path, "w") as f:
#             json.dump(test_metrics, f, indent=2)
#         print(f"  Saved to: {test_metrics_path}")

# gc.collect()
# if torch.cuda.is_available():
#     torch.cuda.empty_cache()
# print("CELL 9 DONE\n")

## Cell 10: Inference Example

In [ ]:
# print("=" * 60)
# print("CELL 10: INFERENCE EXAMPLE")
# print("=" * 60)

# # Use the best model (already loaded by Trainer)
# inference_model = trainer.model
# # Unwrap DataParallel if present
# if hasattr(inference_model, 'module'):
#     inference_model = inference_model.module

# device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
# inference_model = inference_model.to(device)
# inference_model.eval()

# # Switch to left padding for generation
# tokenizer.padding_side = "left"

# # Example product
# product_name = "Áo thun nam thể thao chạy bộ thoáng khí PROMAX"
# description = (
#     "Chất liệu vải mè siêu nhẹ, co giãn 4 chiều, thấm hút mồ hôi. "
#     "Thiết kế basic, logo phản quang. Phù hợp cho mọi hoạt động thể thao "
#     "và mặc hàng ngày. Size S M L XL XXL."
# )

# user_content = f"tạo quảng cáo cho sản phẩm sau:\nTên sản phẩm: {product_name}\nMô tả: {description}"
# prompt = f"<|im_start|>user\n{user_content}<|im_end|>\n<|im_start|>assistant\n"

# print(f"Prompt:\n{prompt}\n")

# inputs = tokenizer(
#     prompt,
#     return_tensors="pt",
#     padding=False,
#     truncation=True,
#     max_length=MAX_SEQ_LENGTH - 1024,
#     add_special_tokens=False,
# ).to(device)

# eos_ids = [tokenizer.eos_token_id]
# im_end_id_val = tokenizer.convert_tokens_to_ids("<|im_end|>")
# if im_end_id_val != tokenizer.unk_token_id and im_end_id_val not in eos_ids:
#     eos_ids.append(im_end_id_val)

# with torch.no_grad():
#     output_ids = inference_model.generate(
#         **inputs,
#         max_new_tokens=1024,
#         num_beams=5,
#         no_repeat_ngram_size=3,
#         early_stopping=True,
#         temperature=0.7,
#         top_k=50,
#         top_p=0.9,
#         pad_token_id=tokenizer.pad_token_id,
#         eos_token_id=eos_ids,
#     )

# input_len = inputs['input_ids'].shape[1]
# gen_ids = output_ids[0][input_len:]
# generated_text = tokenizer.decode(gen_ids, skip_special_tokens=True).strip()

# print(f"--- Generated Advertisement ---")
# print(generated_text)
# print(f"--- End ---")

# # Restore padding side
# tokenizer.padding_side = "right"

# print("\n" + "=" * 60)
# print("ALL CELLS COMPLETE!")
# print("=" * 60)
# print(f"\nFinal model saved at: {FINAL_MODEL_DIR}")
# print("On Kaggle, commit the notebook to persist the output files.")
# print("Download the adapter from: Output → output/final_best_model/")